# Quadratic local search for the Berge–Fulkerson conjecture

Let \(G\) be a bridgeless cubic graph on \(n\) vertices and let

\[
\mathcal M=(M_1,\ldots,M_6)
\]

be a multiset of six perfect matchings. For each edge, let

\[
m(e)=|\{i:e\in M_i\}|,\qquad d(e)=m(e)-2,
\]

and define

\[
Q(\mathcal M)=\sum_{e\in E(G)}d(e)^2.
\]

A Fulkerson cover is exactly a state with \(Q=0\).

## Quadratic Descent Theorem

1. If \(Q>2n\), one of the six matchings can be replaced by a perfect
   matching so that \(Q\) strictly decreases.
2. Such a replacement is found by six minimum-weight perfect-matching
   computations.
3. Starting from any state, at most \(5n\) strict replacements are needed
   to reach \(Q\le 2n\).
4. If \(G\) is connected, nonbipartite, and cyclically 4-edge-connected,
   every one-replacement local minimum satisfies

\[
Q\le
\begin{cases}
2n-6,&n\equiv0\pmod6,\\
2n-4,&n\equiv2\pmod6,\\
2n-2,&n\equiv4\pmod6.
\end{cases}
\]

Thus the same algorithm reaches this sharper barrier.

In [1]:
import sympy as sp
import pandas as pd

n, Q = sp.symbols("n Q", integer=True, nonnegative=True)
d = sp.symbols("d", integer=True)

remove_delta = sp.expand((d - 1)**2 - d**2)
add_after_removal = sp.expand(d**2 - d**2)  # placeholder display only

print("Removing an incident matching edge changes d^2 by:", remove_delta)

Removing an incident matching edge changes d^2 by: 1 - 2*d


For a fixed index \(i\), define

\[
w_i(e)=2d(e)+1-2\mathbf 1_{e\in M_i}.
\]

If \(M_i\) is replaced by a perfect matching \(N\), then

\[
Q(\mathcal M-M_i+N)-Q(\mathcal M)
=
w_i(N)-w_i(M_i).
\]

Hence the best one-coordinate rewrite is a minimum-weight perfect matching.

In [2]:
# Verify the edgewise replacement identity for the four binary membership cases.
I, J = sp.symbols("I J", integer=True)
for i_value in (0, 1):
    for j_value in (0, 1):
        direct = sp.expand(
            (d - i_value + j_value)**2 - d**2
        )
        proposed = sp.expand(
            i_value * (-2*d + 1)
            + j_value * (2*d + 1 - 2*i_value)
        )
        assert sp.expand(direct - proposed) == 0

print("Edgewise replacement identity verified in all four binary cases.")

Edgewise replacement identity verified in all four binary cases.


Let

\[
S_i=\sum_{e\in M_i}d(e).
\]

The all-\(1/3\) vector belongs to the perfect-matching polytope of every
bridgeless cubic graph. Therefore

\[
\min_N w_i(N)\le \frac13\sum_e w_i(e)=\frac n6.
\]

Also,

\[
w_i(M_i)=2S_i-\frac n2.
\]

At a one-replacement local minimum, \(M_i\) is itself minimum weight, so

\[
2S_i-\frac n2\le\frac n6,
\qquad S_i\le\frac n3.
\]

Finally,

\[
\sum_{i=1}^6 S_i
=
\sum_e m(e)(m(e)-2)
=
Q,
\]

because \(\sum_e(m(e)-2)=0\). Thus \(Q\le2n\).

In [3]:
S = sp.symbols("S", real=True)
assert sp.simplify(
    sp.solve(sp.Eq(2*S - n/2, n/6), S)[0] - n/3
) == 0

m = sp.symbols("m")
assert sp.expand(
    m*(m-2) - ((m-2)**2 + 2*(m-2))
) == 0

print("Coordinate bound and summation identity verified.")

Coordinate bound and summation identity verified.


## Maximum possible value and iteration bound

At each vertex, the three incident multiplicities are nonnegative integers
summing to six. The maximum of

\[
(a-2)^2+(b-2)^2+(c-2)^2
\]

is \(24\), attained at \((6,0,0)\). Each edge is counted at two endpoints,
so

\[
Q\le12n.
\]

The potential is even. Every strict step lowers it by at least two.
Consequently, the descent algorithm reaches \(Q\le2n\) in at most

\[
\frac{12n-2n}{2}=5n
\]

steps.

In [4]:
triples = [
    (a, b, 6-a-b)
    for a in range(7)
    for b in range(7-a)
]
values = [
    ((a-2)**2 + (b-2)**2 + (c-2)**2, (a,b,c))
    for a,b,c in triples
]
maximum = max(values)
print("Maximum vertex contribution:", maximum)
assert maximum[0] == 24

Maximum vertex contribution: (24, (6, 0, 0))


## Strict barrier in the cyclically 4-edge-connected nonbipartite case

Only the case \(3\mid n\) needs extra work. Suppose a local minimum has
\(Q=2n\). Then every coordinate inequality is tight:

\[
S_i=\frac n3.
\]

For each \(i\), the fractional point \(x=\frac13\mathbf1\) is therefore a
minimizer of \(w_i\).

In a connected cyclically 4-edge-connected cubic graph, every 3-edge cut is
trivial. Hence \(x\) lies in the relative interior of the perfect-matching
polytope. A linear functional minimized at a relative-interior point is
constant on that polytope. Since the only affine equations are the vertex
degree equations, there are numbers \(\alpha_v\) such that

\[
w_i(uv)=\alpha_u+\alpha_v.
\]

At every vertex \(v\),

\[
\sum_{e\ni v}w_i(e)=1,
\]

because the incident multiplicities sum to six and exactly one incident edge
lies in \(M_i\). Thus

\[
(3I+A)\alpha=\mathbf1,
\]

where \(A\) is the adjacency matrix.

The constant vector \(\alpha=\frac16\mathbf1\) is a solution. In a connected
nonbipartite cubic graph, \(-3\) is not an eigenvalue of \(A\), so the solution
is unique. It follows that every edge has weight

\[
w_i(e)=\frac13,
\]

contradicting the fact that all \(w_i(e)\) are odd integers.

Therefore equality \(Q=2n\) is impossible. Combining this with integrality
gives the residue-class barriers stated above.

In [5]:
def sharp_barrier(n_value):
    if n_value % 2:
        raise ValueError("A cubic graph has even order.")
    if n_value % 6 == 0:
        return 2*n_value - 6
    if n_value % 6 == 2:
        return 2*n_value - 4
    return 2*n_value - 2

table = pd.DataFrame(
    {
        "n mod 6": [0, 2, 4],
        "local-minimum upper bound": ["2n-6", "2n-4", "2n-2"],
    }
)
table

,n mod 6,local-minimum upper bound
0,0,2n-6
1,2,2n-4
2,4,2n-2


## Algorithm

At each iteration:

1. compute multiplicities \(m(e)\) and \(Q\);
2. for each \(i\), assign weights
   \[
   w_i(e)=2(m(e)-2)+1-2\mathbf1_{e\in M_i};
   \]
3. compute a minimum-weight perfect matching \(N_i\);
4. replace the coordinate giving the largest strict decrease.

Minimum-weight perfect matching is polynomial-time. The number of strict
iterations is linear in \(n\), so normalization to the explicit low-defect
region is polynomial-time.

The theorem does not prove Berge–Fulkerson: local minima can remain below the
barrier. It rigorously confines every obstruction to a linearly bounded,
arithmetically structured region.